# Градиентный бустинг (Gradient Boosting)

Мы познакомились с **бэггингом** – это способ построения ансамбля деревьев решений, обученных независимо друг от друга. С одной стороны это удобно тем, что можно процесс обучения распараллелить. Но с другой стороны, для решения сложных задач требуется очень большое количество деревьев.

В этом блокноте речь пойдет о **бустинге**. При этом способе деревья строятся не параллельно (как в случайном лесе), а последовательно, одно за другим. При этом каждое следующее дерево исправляет ошибки всех предыдущих. Деревья решений в этом случае могут быть очень простые, то есть неглубокие.

<div>
<img src="https://drive.google.com/uc?export=view&id=140OQVxGP79UuOL3z5fvOjyQQPEgdWCBp" width="500"/>
</div>

Каждый базовый алгоритм из ансамбля (дерево решений) дает прогноз $b(x)$ для объекта $x$. Тогда ответ ансамбля - это сумма ответов базовых алгоритмов:

$$a_M(x)=\sum_{i=0}^Mb_i(x)$$

где $M$ - количество базовых алгоритмов, входящих в ансамбль.

Так как ансамбль строится итеративно, нужно вначале обучить самый первый простой алгоритм. Здесь можно брать обычные константные алгоритмы. Например, для задачи регрессии

$$b_{0} = \frac{1}{n}\sum_{i=1}^{n}y_{i},$$

в случае классификации - самый часто встречающийся в выборке класс:

$$b_{0} = \underset{y}{\text{argmax}}\sum_{i=1}^{n}[y_{i} = y].$$

Тогда ответ ансамбля будет

$$a_0(x)=b_0$$

Далее для каждой итерации $t=1,...,M$ вычисляется вектор сдвига $s$ для всех объектов (это ошибки ансамбля):

$$s_t=-\begin{bmatrix}
\frac{\partial Loss}{\partial a_{t-1}(x_{i})}
\end{bmatrix}, \text{ for } i=1,...,n$$

и базовый алгоритм обучается на ошибках, полученных ансаблем на предыдущих итерациях:

$$b_{t}(x) = {\text{argmin}}\sum_{i=1}^{n}Loss(s_t(x_i), b(x_i))$$

Полученные прогнозы добавляются к прогнозам всех предыдущих базовых алгоритмов:

$$a_t(x)=a_{t-1}(x)+\eta b_{t}(x)=\sum_{i=0}^{t}b_i(x),$$

где $\eta$ - скорость обучения.



In [1]:
# задача регрессии
import numpy as np

y_true = np.array([35, 50, 65])

In [2]:
# 0 base: mean(y)
y_pred_0 = [np.mean(y_true)] * len(y_true)
print('y_pred_0 =\t', np.array(y_pred_0))
y_pred_ensemble = np.array(y_pred_0)
print('y_ensemble =\t', y_pred_ensemble)
s_1 = y_true - y_pred_0
print('bias_1 =\t', s_1)

y_pred_0 =	 [50. 50. 50.]
y_ensemble =	 [50. 50. 50.]
bias_1 =	 [-15.   0.  15.]


In [3]:
# 1 tree: mean(y_pred_1 - s_1)**2 -> min
y_pred_1 = np.array([-10, -1, 7])
print('y_pred_1 =\t', np.array(y_pred_1))
y_pred_ensemble = y_pred_0 + y_pred_1
print('y_ensemble =\t', y_pred_ensemble)
s_2 = s_1 - y_pred_1 # = y_true - y_pred_0 - y_pred_1
print('bias_2 =\t', s_2)

y_pred_1 =	 [-10  -1   7]
y_ensemble =	 [40. 49. 57.]
bias_2 =	 [-5.  1.  8.]


In [4]:
# 2 tree: mean(y_pred_3 - s_2)**2 -> min
y_pred_2 = np.array([-2, 2, 10], dtype='float')
print('y_pred_2 =\t', np.array(y_pred_2))
y_pred_ensemble = y_pred_0 + y_pred_1 + y_pred_2
print('y_ensemble =\t', y_pred_ensemble)
print('y_true =\t', np.array(y_true, dtype='float'))
s_3 = s_2 - y_pred_2 # = y_true - y_pred_0 - y_pred_1 - y_pred_2
print('bias_3 =\t', s_3)

y_pred_2 =	 [-2.  2. 10.]
y_ensemble =	 [38. 51. 67.]
y_true =	 [35. 50. 65.]
bias_3 =	 [-3. -1. -2.]


### Функции потерь

Это функции, которые оптимизируется (минимизируется) внутри алгоритма.

#### Функции потерь регрессии

Наиболее частые варианты:

- **$L_2$ loss**, она же *Gaussian loss*. Это самый частый и простой вариант: $L_2(y, \hat y)=(y - \hat y)^2$.
- **$L_1$ loss**, она же *Laplacian loss*: $L_1(y, \hat y)=|y - \hat y|$. Эта, на первый взгляд, не очень дифференцируемая вещь, но в некоторых задачах эта функция потерь предпочтительнее, так как она не так сильно штрафует большие отклонения, нежели квадратичная функция.
- **$L_q$ loss**, она же *Quantile loss*:
$$L_q(y,\hat y)=\begin{cases}
    (1-α)\cdot |y-\hat y|,      & \quad \text{if } y-\hat y \le 0\\
    α\cdot |y-\hat y|,  & \quad \text{if } y-\hat y > 0
  \end{cases}$$

  $α \in (0,1)$ - задаваемый квантиль.  
  Эта функция ассиметрична и больше штрафует наблюдения, оказывающиеся по нужную нам сторону квантили.
- **$L_{\delta}$ loss** - функция потерь Хьюбера, она же *Huber loss*:
$$L_{\delta}(y,\hat y)=\begin{cases}
    \frac{1}{2} (y-\hat y)^2,      & \quad \text{if } |y-\hat y| \le \delta\\
    \delta \cdot (|y-\hat y|-\frac{1}{2}\delta),  & \quad \text{otherwise }
  \end{cases}$$

  $\delta$ - задаваемый параметр.  
  Эта функция квадратична для малых значений ошибки, и линейна для больших значений, то есть менее чувствительна к выбросам, чем $L_2$ loss.

#### Функции потерь классификации

Разберем бинарную классификацию, когда $y \in \left\{-1, 1\right\}$. Наиболее известные варианты таких классификационных функций потерь:

- $L(y, \hat y)=log(1+exp(-2y\hat y))$ - *Logistic loss*, она же *Bernoulli loss*. Интересное свойство заключается в том, что мы штрафуем даже корректно предсказанные метки классов.  Это самая стандартная и часто используемая функция потерь в бинарной классификации.
- $L(y, \hat y)=exp(-y\hat y)$ - *Adaboost loss*.

Бустинги на сегодняшний день являются самыми сильными классическими алгоритмами машинного обучения. Можно встретить несколько реализаций бустинга, например, в библиотеках Xgboost, Catboost или Lightgbm. Мы же возьмем градиентный бустинг из уже знакомой нам библиотеки scikit-learn, где алгоритм представлен двумя классами `GradientBoostingClassifier` (для классификации) и `GradientBoostingRegressor` (для регрессии).



## Генерация данных

Сформируем с помощью `sklearn.make_classification` датасет из 1000 объектов с двумя признаками.

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.model_selection import KFold, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import r2_score, accuracy_score

In [7]:
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size': 12, 'figure.figsize': (5,4)})
sns.set(style='whitegrid')
sns.set_context("paper", font_scale=1.5)
pd.options.display.float_format = '{:,.2f}'.format
pd.set_option('display.max_rows', 50)

In [8]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000,
                           n_features=2, n_informative=2,
                           n_classes=2, n_redundant=0,
                           n_clusters_per_class=2, flip_y=0.1, random_state=1)

## GradientBoostingClassifier из библиотеки sklearn

In [9]:
from sklearn.ensemble import GradientBoostingClassifier

boost = GradientBoostingClassifier(random_state=42)
boost.get_params()

{'ccp_alpha': 0.0,
 'criterion': 'friedman_mse',
 'init': None,
 'learning_rate': 0.1,
 'loss': 'log_loss',
 'max_depth': 3,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 100,
 'n_iter_no_change': None,
 'random_state': 42,
 'subsample': 1.0,
 'tol': 0.0001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

### Основные параметры градиентного бустинга

* `loss`: {‘log_loss’, ‘exponential’}, default=’log_loss’ - оптимизируемая функция
* `learning_rate`, default=0.1  - скорость обучения
* `n_estimators`, default=100 – число деревьев / количество итераций
* `subsample`, default=1.0 - доля объектов, на которых обучается один базовый алгоритм. Если `subsample < 1.0`, то имеем Stochastic Gradient Boosting
* `max_depth`, default=3 – максимальная глубина дерева
* `max_features`: {None, ‘sqrt’, ‘log2’}, int or float – максимальное число признаков, по которым ищется лучшее разбиение в узле
* `min_samples_leaf`, default=1 – минимальное число объектов в листе
* `min_samples_split`, default=2 – минимальное количество объектов, необходимое для разделения в узле
* `tol`, default=1e-4 - критерий останова. Если в течение `n_iter_no_change` итераций ошибка меняется на значение, меньшее, чем `tol`, то обучение останавливается

## Оценка точности алгоритма

Модель, обладающая хорошей **обобщающей способностью**, способна предсказывать примерно на одном и том же уровне точности как на обучающем датасете, так и на новых данных.

В процессе построения модели необходимо оценивать качество прогнозов на данных, которые модель не видела во время обучения, чтобы избежать недообучения / переобучения алгоритма.

Для этого используют два приёма - *отложенную выборку* и *кросс-валидацию*.


**Отложенная выборка**

Обучающая выборка разбивается на две части train и test (valid), в тестовую часть, как правило, идёт от 20% до 33% наблюдений, остальное в обучающую. На обучающей части мы тренируем нашу модель и настраиваем гиперпараметры, а после сравниваем прогнозы между обучающей и тестовой частями, чтобы понять, как модель поведёт себя на данных, которые не видела до этого.

**Кросс-валидация**

Данный способ в своей основе похож на предыдущий. Обучающая выборка разбивается на $k$ непересекающихся частей, как правило 3 или 5, после чего каждая из частей поочерёдно выступает в роли тестовой части, а оставшиеся в роли обучающей. Таким образом модель проходит $k$ стадий проверок, полностью задействуя обучающий датасет.

<div>
<img src="https://drive.google.com/uc?export=view&id=1q3RKP0rBDq8FCm7LO6kX0zHqzCYZWDcc" width="500"/>
</div>

Проверим точность градиентного бустинга, используя кросс-валидацию.

In [10]:
scores = cross_val_score(boost, X, y, cv=10, scoring='accuracy')
print(scores)
print(f'{scores.mean():.3f}')

[0.78 0.82 0.83 0.81 0.86 0.81 0.78 0.84 0.8  0.82]
0.815


## Подбор гиперпараметров градиентного бустинга по сетке

Метод поиска по сетке (Grid Search) принимает на вход модель и различные значения гиперпараметров (сетку гиперпараметров). Далее, для каждого возможного сочетания значений гиперпараметров метод считает ошибку и в конце выбирает сочетание, при котором ошибка минимальна.

In [11]:
from sklearn.model_selection import GridSearchCV

### Параметры GridSearchCV из библиотеки scikit-learn

- `estimator` — модель, которую будем обучать
- `param_grid` — словарь с параметрами, которые хотим подобрать
- `cv` — на сколько фолдов делим датасет
- `scoring` — метрика для оценки точности модели

In [12]:
params = { 'n_estimators': np.arange(10, 101, 10),
           'max_depth': [1, 2, 3]
         }

In [13]:
grid = GridSearchCV(boost, params, scoring='accuracy', cv=3)
grid.fit(X, y)

GridSearchCV(cv=3, estimator=GradientBoostingClassifier(random_state=42),
             param_grid={'max_depth': [1, 2, 3],
                         'n_estimators': array([ 10,  20,  30,  40,  50,  60,  70,  80,  90, 100])},
             scoring='accuracy')

In [14]:
# атрибуты GridSearchCV
print(grid.best_estimator_)
print(grid.best_params_)
print(f'{grid.best_score_:.3f}')

GradientBoostingClassifier(n_estimators=np.int64(50), random_state=42)
{'max_depth': 3, 'n_estimators': np.int64(50)}
0.824


In [15]:
pred = grid.predict(X)
accuracy_score(y, pred)

0.868

## Дополнительный материал

[GradientBoostingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html#sklearn.ensemble.GradientBoostingRegressor)

[GradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html#sklearn.ensemble.GradientBoostingClassifier)

[The scoring parameter](https://scikit-learn.org/stable/modules/model_evaluation.html#scoring-parameter)

## Задания

1. Попробуйте улучшить точность (accuracy) градиентного бустинга на данных из урока (2 балла).

In [16]:
# код

2. Загрузите датасет `prepared.csv`. Выполните следующее (10 баллов):

- Выделите отдельно матрицу с признаками `X` (это датасет `df` без столбцов `'Id'`, `'Price'` и `'PriceOneRoom'`) и массив с ответами `y = df['Price']`.
- Обучите модель градиентного бустинга `GradientBoostingRegressor` и проверьте ее точность, используя кросс-валидацию.
- Найдите самые важные для модели признаки.  
- Подберите такие гиперпараметры модели, чтобы коэффициент детерминации `r2` был больше 0.73.

In [17]:
!gdown 1WonVHHMWLsLLE9sdKKTKPA-WfZTUUEra
df = pd.read_csv('/content/prepared.csv', index_col=0)

Downloading...
From: https://drive.google.com/uc?id=1WonVHHMWLsLLE9sdKKTKPA-WfZTUUEra
To: /content/prepared.csv
100% 1.64M/1.64M [00:00<00:00, 141MB/s]


In [18]:
# код